# 02 VLM Judge Analysis
Qwen2.5-VL-7B scene and detection quality assessment.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import random
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from IPython.display import display
from pycocotools.coco import COCO

from src.detector import DetectorWrapper
from src.vlm_judge import VLMJudge
from src.utils import load_image

DATA_DIR = Path("../data/coco")
VAL_DIR = DATA_DIR / "val2017"
ANN_FILE = DATA_DIR / "annotations" / "instances_val2017.json"
CACHE_FILE = Path("../data/vlm_cache_20images.json")

assert DATA_DIR.exists(), f"Missing COCO data directory: {DATA_DIR.resolve()}"
assert VAL_DIR.exists(), f"Missing COCO val2017 images: {VAL_DIR.resolve()}"
assert ANN_FILE.exists(), f"Missing COCO annotations: {ANN_FILE.resolve()}"

coco_val = COCO(str(ANN_FILE))
img_ids = coco_val.getImgIds()

print(f"COCO val images : {len(img_ids):,}")
print(f"COCO data dir   : {DATA_DIR.resolve()}")
print(f"Cache file      : {CACHE_FILE.resolve()}")

## 1. Load Detector and VLM Judge
⚠️ VLM requires ~4.5 GB VRAM (INT4).

In [ ]:
import torch

def vram_allocated_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1e9

detector = DetectorWrapper(model_name="yolox-s", config_path="../config/detector_config.yaml")
detector.load_model()
detector_vram = vram_allocated_gb()
print(f"Detector loaded: {detector.model_name} on {detector.device}")
print(f"VRAM after detector load: {detector_vram:.2f} GB")

judge = VLMJudge(config_path="../config/vlm_config.yaml")
judge.load_model()
combined_vram = vram_allocated_gb()
print(f"VLM Judge loaded: {judge.model_name}")
print(f"VRAM after VLM load: {combined_vram:.2f} GB")
print(f"Estimated combined model VRAM: {combined_vram:.2f} GB")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA unavailable. VLM inference may be extremely slow or unsupported on this machine.")

## 2. Run VLM on 20 COCO Images
Results cached to avoid re-running.

In [ ]:
SEED = 42
MAX_IMAGES = 20
random.seed(SEED)

def detection_to_dict(det):
    return {
        "detection_id": int(det.detection_id),
        "bbox": [int(v) for v in det.bbox],
        "class_id": int(det.class_id),
        "class_name": str(det.class_name),
        "confidence": float(det.confidence),
        "area": int(det.area),
        "relative_size": str(det.relative_size),
    }

def assessment_to_dict(assessment):
    return {
        "detection_id": int(assessment.detection_id),
        "quality_score": float(assessment.quality_score),
        "scene_complexity": str(assessment.scene_complexity),
        "object_relative_size": str(assessment.object_relative_size),
        "is_false_positive": bool(assessment.is_false_positive),
        "reasoning": str(assessment.reasoning),
    }

if CACHE_FILE.exists():
    with CACHE_FILE.open("r", encoding="utf-8") as handle:
        records = json.load(handle)
    print(f"Loaded from cache: {CACHE_FILE}")
else:
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    selected_img_ids = random.sample(img_ids, k=min(MAX_IMAGES, len(img_ids)))
    records = []

    for index, image_id in enumerate(selected_img_ids, start=1):
        info = coco_val.loadImgs(image_id)[0]
        image_path = VAL_DIR / info["file_name"]
        image = load_image(str(image_path))
        detections = detector.detect(image)

        t0 = time.time()
        assessments = judge.assess_detections(image, detections)
        latency = time.time() - t0

        record = {
            "image_id": int(image_id),
            "filename": info["file_name"],
            "num_detections": int(len(detections)),
            "latency": float(latency),
            "detections": [detection_to_dict(det) for det in detections],
            "assessments": [assessment_to_dict(assessment) for assessment in assessments],
        }
        records.append(record)
        print(f"Image {index}/{len(selected_img_ids)}: {info['file_name']} — {len(detections)} dets, {latency:.1f}s")

    with CACHE_FILE.open("w", encoding="utf-8") as handle:
        json.dump(records, handle, indent=2)
    print(f"Saved to cache: {CACHE_FILE}")

total_images = len(records)
total_assessments = sum(len(record.get("assessments", [])) for record in records)
total_detections = sum(int(record.get("num_detections", 0)) for record in records)
print(f"\nSummary:")
print(f"  total images             : {total_images}")
print(f"  total detections         : {total_detections}")
print(f"  total detections assessed: {total_assessments}")

## 3. Scene Complexity Distribution

In [ ]:
complexity_order = ["low", "medium", "high"]
image_complexities = []

for record in records:
    values = [a.get("scene_complexity", "medium") for a in record.get("assessments", [])]
    if values:
        complexity = Counter(values).most_common(1)[0][0]
    else:
        complexity = "medium"
    image_complexities.append(complexity)
    record["image_scene_complexity"] = complexity

complexity_counts = Counter(image_complexities)
counts = [complexity_counts.get(label, 0) for label in complexity_order]
percentages = [100.0 * count / max(len(image_complexities), 1) for count in counts]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].pie(counts, labels=complexity_order, autopct="%1.1f%%", startangle=90, colors=["green", "orange", "red"])
axes[0].set_title("Scene Complexity Share")
axes[1].bar(complexity_order, counts, color=["green", "orange", "red"])
axes[1].set_ylabel("image count")
axes[1].set_title("Scene Complexity Counts")
plt.tight_layout()
plt.show()

complexity_df = pd.DataFrame({
    "scene_complexity": complexity_order,
    "count": counts,
    "percentage": percentages,
})
display(complexity_df)

## 4. Quality Score Distribution

In [ ]:
rows = []
for record in records:
    detections_by_id = {int(det["detection_id"]): det for det in record.get("detections", [])}
    for assessment in record.get("assessments", []):
        det = detections_by_id.get(int(assessment["detection_id"]))
        if det is None:
            continue
        rows.append({
            "image_id": record["image_id"],
            "filename": record["filename"],
            "detection_id": int(assessment["detection_id"]),
            "bbox": det["bbox"],
            "class_name": det["class_name"],
            "confidence": float(det["confidence"]),
            "quality_score": float(assessment["quality_score"]),
            "scene_complexity": assessment["scene_complexity"],
            "object_relative_size": assessment["object_relative_size"],
            "is_false_positive": bool(assessment["is_false_positive"]),
            "reasoning": assessment["reasoning"],
        })

analysis_df = pd.DataFrame(rows)
display(analysis_df.head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(analysis_df["quality_score"], bins=20, color="tab:blue", alpha=0.85)
axes[0].set_xlabel("VLM quality_score")
axes[0].set_ylabel("detection count")
axes[0].set_title("Quality Score Distribution")

color_map = {"low": "green", "medium": "orange", "high": "red"}
for complexity in ["low", "medium", "high"]:
    subset = analysis_df[analysis_df["scene_complexity"] == complexity]
    axes[1].scatter(
        subset["confidence"], subset["quality_score"],
        label=complexity, color=color_map[complexity], alpha=0.7
    )
axes[1].set_xlabel("detector confidence")
axes[1].set_ylabel("VLM quality_score")
axes[1].set_title("Detector Confidence vs VLM Quality")
axes[1].legend(title="scene")
axes[1].grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

if len(analysis_df) >= 2:
    r_value = float(np.corrcoef(analysis_df["confidence"], analysis_df["quality_score"])[0, 1])
else:
    r_value = float("nan")
print(f"Pearson r(confidence, quality_score): {r_value:.3f}")
if np.isfinite(r_value):
    if abs(r_value) >= 0.5:
        print("Quality scores show a strong relationship with detector confidence in this sample.")
    elif abs(r_value) >= 0.25:
        print("Quality scores show a moderate relationship with detector confidence in this sample.")
    else:
        print("Quality scores are only weakly correlated with detector confidence in this sample.")

## 5. VLM False Positive Flags

In [ ]:
fp_counts = analysis_df["is_false_positive"].value_counts().reindex([False, True], fill_value=0)

def xywh_to_xyxy(bbox):
    x, y, w, h = bbox
    return [float(x), float(y), float(x + w), float(y + h)]

def iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union

cat_name_to_id = {cat["name"]: cat["id"] for cat in coco_val.loadCats(coco_val.getCatIds())}
gt_cache = {}
for image_id in analysis_df["image_id"].unique():
    ann_ids = coco_val.getAnnIds(imgIds=int(image_id))
    gt_cache[int(image_id)] = coco_val.loadAnns(ann_ids)

matches_gt = []
best_ious = []
for _, row in analysis_df.iterrows():
    category_id = cat_name_to_id.get(row["class_name"])
    gt_anns = [ann for ann in gt_cache.get(int(row["image_id"]), []) if ann["category_id"] == category_id]
    det_box = [float(v) for v in row["bbox"]]
    ious = [iou_xyxy(det_box, xywh_to_xyxy(ann["bbox"])) for ann in gt_anns]
    best_iou = max(ious) if ious else 0.0
    best_ious.append(best_iou)
    matches_gt.append(best_iou >= 0.5)

analysis_df = analysis_df.copy()
analysis_df["best_coco_iou"] = best_ious
analysis_df["matches_coco_gt"] = matches_gt
analysis_df["vlm_fp_correct_vs_coco"] = analysis_df["is_false_positive"] == (~analysis_df["matches_coco_gt"])

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.bar(["False", "True"], [fp_counts[False], fp_counts[True]], color=["tab:green", "tab:red"])
ax.set_xlabel("is_false_positive")
ax.set_ylabel("detection count")
ax.set_title("VLM False Positive Flags")
plt.tight_layout()
plt.show()

summary_df = (
    analysis_df.groupby("is_false_positive")
    .agg(
        count=("detection_id", "count"),
        mean_confidence=("confidence", "mean"),
        mean_quality_score=("quality_score", "mean"),
    )
    .reset_index()
)
display(summary_df)

fp_correctness_df = pd.crosstab(
    analysis_df["is_false_positive"],
    analysis_df["matches_coco_gt"],
    rownames=["VLM flagged FP"],
    colnames=["Matches COCO GT IoU>=0.5"],
)
print("\nVLM FP flag vs COCO annotation match:")
display(fp_correctness_df)
accuracy_vs_coco = float(analysis_df["vlm_fp_correct_vs_coco"].mean()) if len(analysis_df) else float("nan")
print(f"VLM FP flag agreement with COCO IoU>=0.5 heuristic: {accuracy_vs_coco:.3f}")

print("| is_false_positive | count | mean confidence | mean quality_score |")
print("|---|---:|---:|---:|")
for _, row in summary_df.iterrows():
    print(
        f"| {bool(row['is_false_positive'])} | {int(row['count'])} | "
        f"{row['mean_confidence']:.3f} | {row['mean_quality_score']:.3f} |"
    )

fp_examples = analysis_df[analysis_df["is_false_positive"]].head(3)
print("\nExample VLM reasoning strings for FP-flagged detections:")
if fp_examples.empty:
    print("  No false-positive detections were flagged in this cached sample.")
else:
    for _, row in fp_examples.iterrows():
        print(f"  - {row['filename']} #{int(row['detection_id'])} {row['class_name']}: {row['reasoning']}")

## 6. VLM Inference Latency

In [ ]:
latencies = np.asarray([float(record.get("latency", 0.0)) for record in records], dtype=np.float32)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.hist(latencies, bins=min(10, max(len(latencies), 1)), color="tab:purple", alpha=0.85)
ax.set_xlabel("latency seconds / image")
ax.set_ylabel("image count")
ax.set_title("VLM Inference Latency")
plt.tight_layout()
plt.show()

print("Latency summary:")
print(f"  mean  : {latencies.mean():.2f}s")
print(f"  std   : {latencies.std():.2f}s")
print(f"  min   : {latencies.min():.2f}s")
print(f"  max   : {latencies.max():.2f}s")
print(f"  median: {np.median(latencies):.2f}s")
print("\nVLM runs once per image regardless of detection count. Bottleneck is GPU prefill time for the image tokens.")

## 7. Qualitative: Image + Assessments

In [ ]:
def pick_record(predicate, used_ids):
    for record in records:
        if record["image_id"] in used_ids:
            continue
        if predicate(record):
            used_ids.add(record["image_id"])
            return record
    return None

used_ids = set()
selected_records = []
for complexity in ["low", "medium", "high"]:
    record = pick_record(lambda r, c=complexity: r.get("image_scene_complexity") == c, used_ids)
    if record is not None:
        selected_records.append(record)

fp_record = pick_record(lambda r: any(a.get("is_false_positive", False) for a in r.get("assessments", [])), used_ids)
if fp_record is not None:
    selected_records.append(fp_record)

for record in records:
    if len(selected_records) >= 4:
        break
    if record["image_id"] not in used_ids:
        used_ids.add(record["image_id"])
        selected_records.append(record)

for record in selected_records[:4]:
    image = load_image(str(VAL_DIR / record["filename"]))
    detections_by_id = {int(det["detection_id"]): det for det in record.get("detections", [])}

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.2, 1.0]})
    ax_img, ax_text = axes
    ax_img.imshow(image)
    ax_img.set_title(f"{record['filename']} ({record['num_detections']} dets)")
    ax_img.axis("off")

    for det in record.get("detections", []):
        x1, y1, x2, y2 = det["bbox"]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
        ax_img.add_patch(rect)
        ax_img.text(
            x1, max(y1 - 4, 0), f"#{det['detection_id']} {det['class_name']} {det['confidence']:.2f}",
            color="lime", fontsize=8, fontweight="bold",
            bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none")
        )

    lines = ["id | class | conf | q | scene | FP | reasoning", "---|---|---:|---:|---|---|---"]
    for assessment in record.get("assessments", []):
        det = detections_by_id.get(int(assessment["detection_id"]), {})
        reasoning = str(assessment.get("reasoning", ""))[:80]
        lines.append(
            f"{assessment['detection_id']} | {det.get('class_name', '?')} | "
            f"{float(det.get('confidence', 0.0)):.2f} | {assessment['quality_score']:.2f} | "
            f"{assessment['scene_complexity']} | {assessment['is_false_positive']} | {reasoning}"
        )

    ax_text.axis("off")
    ax_text.set_title("VLM assessments")
    ax_text.text(0.0, 1.0, "\n".join(lines), va="top", ha="left", fontsize=8, family="monospace")
    plt.tight_layout()
    plt.show()

## 8. Cleanup

In [ ]:
detector.unload_model()
judge.unload_model()
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Models unloaded.")